# Hello, `pyquist`!

A tour of the core library. We'll synthesize sound from scratch, load and transform audio, build scores and instruments, render a MIDI file, and play a real-time instrument.

> Run cells with **Shift+Enter**.

In [ ]:
# On Google Colab, install pyquist (and its PortAudio system dependency) on
# first run. Locally, this branch is skipped.
import sys

if "google.colab" in sys.modules:
    !apt install libportaudio2
    !pip install --upgrade pyquist

import numpy as np

import pyquist as pq

## 1. `Audio`: a numpy array of samples

`pq.Audio` is a thin wrapper around a 2D `numpy.ndarray` of `float32` samples shaped `(num_samples, num_channels)`, plus a `sample_rate` in Hz. Anything you can do to a numpy array, you can do to `audio.samples`.

In [ ]:
# 1 second of a 440 Hz sine wave (concert A).
sr = 44100
t = np.arange(sr) / sr
audio = pq.Audio(0.5 * np.sin(2 * np.pi * 440 * t), sample_rate=sr)

print(
    f"shape={audio.shape}, sample_rate={audio.sample_rate}, peak={audio.peak_amplitude:.3f}"
)
pq.play(audio)

## 2. Loading audio

`pq.Audio.from_file` reads from disk and `pq.Audio.from_url` fetches from the web (both delegate to `libsndfile` via the `soundfile` package).

In [ ]:
from pyquist.paths import TEST_DATA_DIR

riff = pq.Audio.from_file(
    TEST_DATA_DIR / "388954__fullmetaljedi__blues-riff-in-g-nylon.wav"
)
print(riff)
pq.play(riff)

## 2. Visualization

`pq.plot` shows the waveform; `pq.plot_freq` plots the FFT magnitude spectrum; `pq.plot_spec` plots a spectrogram. All three accept `offset` / `duration` to zoom into a region.

In [ ]:
# Zoom into 10 ms
pq.plot(riff, duration=0.01)
pq.plot_freq(riff, duration=0.01)
pq.plot_spec(riff)

## 4. Transforming audio

A handful of `Audio` methods cover the common edits — slice in time (`segment`), change sample rate (`resample`), mix down (`as_mono`, `as_stereo`), level (`normalize`, `clip`). All return new `Audio` objects.

In [ ]:
# Pull out a 3-second clip starting 5 seconds in, then downsample to
# telephone quality.
clip = riff.as_mono().segment(offset=1.0, duration=3.0).resample(8000)
print(f"{clip.duration:.2f}s at {clip.sample_rate} Hz")
pq.plot_spec(clip)
pq.play(clip)

## 5. Scores: musical events → audio

A `Score` is a list of events, each a `(time, kwargs)` tuple: a `time` (in seconds, or in beats if you supply a metronome) and a `kwargs` dict. (Internally each pair is an `Event`, but `Score` converts plain tuples for you.) The *instrument* — a function called as `instrument(**event.kwargs)` — turns those kwargs into audio. An instrument just declares the kwargs it cares about and absorbs the rest with `**kwargs`.

The library deliberately leaves instruments to you. Here's a simple one that turns each event into a pure sine tone with a quick decay.

In [ ]:
from pyquist.helper import pitch_to_frequency
from pyquist.score import BasicMetronome, Score


def sine_instrument(pitch, duration, **kwargs):
    sr = 44100
    t = np.arange(int(duration * sr)) / sr
    freq = pitch_to_frequency(pitch)
    samples = 0.3 * np.sin(2 * np.pi * freq * t) * np.exp(-3 * t)
    return pq.Audio(samples, sample_rate=sr)


# Twinkle Twinkle, Little Star: each (beat, {kwargs}) tuple is one note.
# Score coerces these tuples into Events automatically.
twinkle = Score(
    [
        (0, {"pitch": 60, "duration": 0.5}),  # C
        (1, {"pitch": 60, "duration": 0.5}),
        (2, {"pitch": 67, "duration": 0.5}),  # G
        (3, {"pitch": 67, "duration": 0.5}),
        (4, {"pitch": 69, "duration": 0.5}),  # A
        (5, {"pitch": 69, "duration": 0.5}),
        (6, {"pitch": 67, "duration": 1.0}),  # G
    ]
)

# 120 BPM → 1 beat = 0.5 s.
pq.play(twinkle.render(sine_instrument, metronome=BasicMetronome(bpm=120)))

## 6. MIDI files

`Score.from_midi` parses a MIDI file into a `(score, metronome)` pair. Each note becomes an `Event` whose `time` is in MIDI ticks and whose `kwargs` includes `pitch`, `velocity`, `duration` (seconds), `program`, and `is_drum`.

In [ ]:
# A copy of Ravel's Boléro (public domain) is bundled with pyquist for testing.
from pyquist.paths import TEST_DATA_DIR

score, metronome = pq.Score.from_midi(TEST_DATA_DIR / "ravel_bolero.mid")
total_seconds = metronome.tick_to_seconds(score.end_time)
print(f"{len(score)} notes, {total_seconds:.0f}s long")


# Simple "meta instrument": noise for drum events, sine for pitched ones.
def midi_instrument(is_drum, duration, **kwargs):
    if is_drum:
        sr = 44100
        n = int(duration * sr)
        envelope = np.exp(-30 * np.arange(n) / sr)
        samples = 0.1 * np.random.randn(n).astype(np.float32) * envelope
        return pq.Audio(samples, sample_rate=sr)
    return sine_instrument(duration=duration, **kwargs)


# Render a 20-second clip starting a few seconds in.
clip = score.segment(
    offset=metronome.seconds_to_tick(11),
    duration=metronome.seconds_to_tick(20),
)
pq.play(clip.render(midi_instrument, metronome=metronome))

## 7. Real-time audio with `Stream`

Everything so far renders audio first and plays it after. A `pq.Stream` instead calls your function on one short block of audio at a time (512 samples, about 12 ms) *while* it plays, so the sound can react to you. The callback writes into `block` in place.

Below is a monophonic "piano": each key restarts a decaying sine at a new pitch. `stream.start()` returns right away and the stream keeps playing in the background, so you can click the keys; click **Stop** when you're done.

> This needs a local install with speakers. Colab runs on a remote server with no audio devices.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError(
        "This example needs ipywidgets, which pyquist doesn't install. "
        "Run `pip install ipywidgets`, then restart the kernel."
    ) from None
from IPython.display import display

from pyquist.env import in_colab


class Piano:
    """A monophonic synth: each key press starts an enveloped sine."""

    def __init__(self):
        self.frequency = 0.0
        self.time = 0.0  # Seconds since the last key press.

    def press(self, pitch):
        self.frequency = pitch_to_frequency(pitch)
        self.time = 0.0

    def callback(self, block):
        t = self.time + np.arange(len(block)) / block.sample_rate
        # Envelope: a 10 ms linear attack, then an exponential decay.
        envelope = np.minimum(t / 0.01, 1.0) * np.exp(-3 * t)
        block.samples[:, 0] = envelope * np.sin(2 * np.pi * self.frequency * t)
        self.time += len(block) / block.sample_rate


piano = Piano()
stream = pq.Stream(piano.callback)

keys = []
for name, pitch in zip("CDEFGABC", [60, 62, 64, 65, 67, 69, 71, 72]):
    key = widgets.Button(description=name, layout=widgets.Layout(width="40px"))
    key.on_click(lambda _, pitch=pitch: piano.press(pitch))
    keys.append(key)
stop = widgets.Button(description="Stop")
stop.on_click(lambda _: stream.stop())
display(widgets.HBox(keys + [stop]))

if in_colab():
    print("Streaming needs local audio devices; run this notebook locally to play.")
else:
    stream.start()

## What's next?

See [pyquist.org](https://pyquist.org) for documentation, including:

- `pyquist.cli` — `pyquist play`, `pyquist record`, `pyquist devices` from the terminal
- `pyquist.web.fetch_freesound` — pull samples from [freesound.org](https://freesound.org/) (requires a free API key on first use)
- `pyquist.web.fetch_theorytab` — fetch pop-song melody + chord scores from [TheoryTab](https://www.hooktheory.com/theorytab)

Have fun!